# NanoGPT (Learn)

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
import os
import sys
from pathlib import Path

CWD = os.path.realpath(os.getcwd())
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

DATA_DIR = Path(PARENT_DIR).parent / 'data'

In [3]:
from reader.loader import TextDataset

dataset = TextDataset(DATA_DIR, device=device)

100%|██████████| 5/5 [00:00<00:00, 656.30it/s]


In [ ]:
from src.modules.architecture.ngram_lm import NgramLanguageModel
from reader.preprocess import decode
import torch

BATCH_SIZE = 16
EMBEDDING_SIZE = 64
SEQ_LENGTH = 32
DROPOUT_RATE = 0.2

N_HEADS = 8
N_BLOCKS = 4
N_GROUPS = 4
LR = 1e-3

EVAL_ITER = 100
EVAL_INTERVAL = 100
EPOCH_SIZE = 10000

torch.manual_seed(1337)

model = NgramLanguageModel(vocab_size=dataset.vocab_size, n_heads=N_HEADS, n_groups=N_GROUPS, embedding_size=EMBEDDING_SIZE, seq_length=SEQ_LENGTH, 
                            n_blocks=N_BLOCKS, dropout_rate=DROPOUT_RATE, device=device)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("Number of parameters:")
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

Number of parameters:
0.212825 M parameters


In [5]:
@torch.no_grad()
def estimate_loss(x, y, model):
    losses = torch.zeros(EVAL_ITER)
    for k in range(EVAL_ITER):
        logits, loss = model(x, y)
        losses[k] = loss.item()
    return losses.mean()

In [6]:
for iter in range(EPOCH_SIZE):

    # every once in a while evaluate the loss on train and val sets
    if iter % EVAL_INTERVAL == 0 or iter == EPOCH_SIZE - 1:
        model.eval()
        x_train, y_train = dataset.load_train(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        x_val, y_val = dataset.load_test(batch_size=BATCH_SIZE, context_window_size=SEQ_LENGTH)
        train_losses = estimate_loss(x_train, y_train, model)
        val_losses = estimate_loss(x_val, y_val, model)
        print(f"step {iter}: train loss {train_losses:.4f}, val loss {val_losses:.4f}")
        model.train()

    # sample a batch of data
    x_train, y_train = dataset.load_train(BATCH_SIZE)

    # evaluate the loss
    logits, loss = model(x_train, y_train)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.6745, val loss 4.6668
step 100: train loss 2.5942, val loss 2.5948
step 200: train loss 2.4359, val loss 2.4038
step 300: train loss 2.2259, val loss 2.3016
step 400: train loss 2.1360, val loss 2.1561
step 500: train loss 2.0047, val loss 2.0626
step 600: train loss 1.9620, val loss 2.0441
step 700: train loss 1.8821, val loss 1.9523
step 800: train loss 1.9000, val loss 1.9562
step 900: train loss 1.8300, val loss 1.9003
step 1000: train loss 1.7681, val loss 1.8542
step 1100: train loss 1.7794, val loss 1.8455
step 1200: train loss 1.6692, val loss 1.8741
step 1300: train loss 1.6611, val loss 1.8483
step 1400: train loss 1.6763, val loss 1.8283
step 1500: train loss 1.6832, val loss 1.7707
step 1600: train loss 1.6908, val loss 1.7833
step 1700: train loss 1.6372, val loss 1.8060
step 1800: train loss 1.6577, val loss 1.7580
step 1900: train loss 1.5862, val loss 1.7574
step 2000: train loss 1.5768, val loss 1.7258
step 2100: train loss 1.5193, val loss 1.7666


In [7]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))


cown to stery in that the mother.13. He would plicind givarred in their acturned out, and left him he well, he withs town, good and nothing-stoods fanator instace,
My councest for him awords' and the offoons and guest of husband. Seee at that the elder able the tive, latter he was apprent once; but and went exace and in soles.

KES:
Uperty as itso peasion fancigify that from the you time ordes working, why only, if course. And ESupie or shall debauch to his hathut, interes lady one expletition and was in whose you an amouth was off there sicious, came been in explace to peoson at probably of our discress of sheir since of inher bureliving. Moscoovs. Bitrovic; with his coar tablute must in the epoch into once, and contempt. By a state of his canned and bewhere's  "asquestes a greunded into would not he retened. You frade happy, loveot come, and this. Alyosha notier Zossimea. Hould in the a dauchoous and always was only are nothing side ingitably doond of givince, he had hewed will beau

In [8]:
from reader.preprocess import encode
context = torch.tensor([encode(dataset.stoi, 'fyodor')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

fyodor Pavloving do yoush his mother.

AUs, I.
Fonased, these read soul:
thate out had end theyse to his deal, brour brothers did not not with his receimed his tyle for to end perhath a precised to his new in Romesnif me sorroway he would you fellinget to bus a catac off his canstage fal only secraciously, he repeded in the child upon the lack bin despitiint of the eldy.

3 I that spot you
That was one explaince, come wodor I sit, mug! It antols a tame a lover-and long dith that whith I staudy that every be proses cassed forinnes and hoolid unto some expece, thin which I  haveter. a sord.";  and  it  bequestion  be  begin  man  destan  up language  of  the  ast  an  soul,  It  call  uto  of  heartupining  had  man  one  every  thinking  upon  it  converiar  or  unce  subject  they  lock.——Alyosha's  make the  pewich's  this  arrity  language   a  cloar:   goes.  It would be be  of might never, and  ona  .    But  the  is,  for  it  quilLity, who,  they  sachil  thatsking  that  as  nei

In [9]:
context = torch.tensor([encode(dataset.stoi, 'slab')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

slab"," he distrun  the  old  in   salleday  these  none  one  thou  saill  they  sentence  a worthy  thousain,  this  luettby,  all  over became  of  "these  sentences  by  veing  me  going  used  a breamed his  older,"phered at some of the child and alwardid lived in. He well, with grettles, sia little father we more and thout the finish. Tregulard laterly, he none, so muriusly deep of the intellight of mictuaily lifie and bid withat hocch home conversia, but and opentension atteac the coally,
Kinigentlet stronk to a monastery to had four bored bout in innite very to for a rely unabled requon; yet town (lax "Slabl"YzeVen hold and going at on the morisher, sended not him, but as:
vigorouns expless.(Af When every Mortupide or not out or all sporfect to knovs. Ind early day that you have sautis of Adelaïda Iva L2easnants and disliatary, were sign for but his cheep spond up of his gat wifle succounts entire frighty did incitepened too,
Complity and it was dee sol. More. And if rest wifel

In [10]:
torch.randint(
            100 - 10,
            (16,)
        )

tensor([61,  0, 64,  5, 71, 87, 35, 81, 55, 49, 10, 24, 35, 78,  2, 31])

In [11]:
context = torch.tensor([encode(dataset.stoi, 'slab')], dtype=torch.long, device=device)
idx_cond = context[:, -32:]

In [12]:
idx_cond

tensor([[72, 65, 54, 55]], device='cuda:0')

In [9]:
test = {'one': 1, 'two': 2}
sorted(test.items(), key=lambda x: x[1], reverse=True)

[('two', 2), ('one', 1)]

In [3]:
test = 1
if True and test:
    print('yes')

yes
